In [6]:
import sys
sys.path.insert(0, "/home/ibm-user/shared/safecircle")  
sys.path.insert(0, "/home/ibm-user")                   

from src.messages import score_message
from src.explain import redact, explain
from src.transactions import score_transaction
from src.risk import decide


In [7]:
import pandas as pd

REPO = "/home/ibm-user/shared/safecircle"
messages     = pd.read_csv(f"{REPO}/scenarios/messages.csv")
transactions = pd.read_csv(f"{REPO}/scenarios/transactions.csv")

# Parsed copies of the timestamps for matching; the original text column is left untouched
messages["ts"]     = pd.to_datetime(messages["timestamp"])
transactions["ts"] = pd.to_datetime(transactions["timestamp"])
messages = messages.sort_values("ts")

print(len(messages), "messages,", len(transactions), "transactions")
transactions


3 messages, 3 transactions


,user_id,txn_id,timestamp,amount,merchant,merchant_category,is_new_payee,expected_action,ts
0,rosa,t1,2026-10-17 10:25,1500.0,QuickMart Gift Cards,gift_card,1,hold_and_alert,2026-10-17 10:25:00
1,walter,t2,2026-10-17 11:00,42.0,CVS Pharmacy,pharmacy,0,allow,2026-10-17 11:00:00
2,linda,t3,2026-10-17 15:10,300.0,Wire to J. Smith,wire,1,hold_and_alert,2026-10-17 15:10:00


In [8]:
def find_message(txn_row):
    """The latest message to the same person in the 24 hours before the payment."""
    window = messages[
        (messages["user_id"] == txn_row["user_id"])
        & (messages["ts"] <= txn_row["ts"])
        & (messages["ts"] >= txn_row["ts"] - pd.Timedelta(hours=24))
    ]
    if len(window) == 0:
        return ""  # no message: score an empty text instead of skipping the payment
    return window.iloc[-1]["text"]

transactions["message_text"] = transactions.apply(find_message, axis=1)
transactions[["user_id", "amount", "merchant", "message_text"]]

,user_id,amount,merchant,message_text
0,rosa,1500.0,QuickMart Gift Cards,"Grandma it's me, I'm in trouble. Please buy gi..."
1,walter,42.0,CVS Pharmacy,Your prescription is ready for pickup.
2,linda,300.0,Wire to J. Smith,Your bank account is locked. Verify your ident...


In [11]:
# How each merchant category reads in a text message to family
CATEGORY_WORDS = {
    "gift_card": "gift-card purchase",
    "wire": "wire transfer",
    "money_transfer": "money transfer",
}

def family_alert(txn_row):
    """The text the trusted family contact receives, built from the transaction row."""
    name = str(txn_row["user_id"]).capitalize()
    amount = f"${float(txn_row['amount']):,.0f}"
    category = txn_row["merchant_category"]
    what = CATEGORY_WORDS.get(category, f"{category.replace('_', ' ')} purchase")
    return (f"SafeCircle paused a {amount} {what} on {name}'s card after a likely scam text. "
            f"Please call {name}.")

# Quick check on Rosa's row
print(family_alert(transactions[transactions["user_id"] == "rosa"].iloc[0]))

SafeCircle paused a $1,500 gift-card purchase on Rosa's card after a likely scam text. Please call Rosa.


In [12]:
TXN_KEYS = ["user_id", "txn_id", "timestamp", "amount", "merchant",
            "merchant_category", "is_new_payee"]

results = []
for _, row in transactions.iterrows():
    msg_result  = score_message(row["message_text"])
    txn_result  = score_transaction(row[TXN_KEYS].to_dict())  # only the contract keys
    decision    = decide(msg_result, txn_result)
    safe_text   = redact(row["message_text"])                 # private details removed BEFORE explain()
    explanation = explain(safe_text, decision)

    results.append({
        "user_id": row["user_id"],
        "risk": decision["risk"],
        "action": decision["action"],
        "expected_action": row["expected_action"],
        "reasons": decision["reasons"],
        "explanation": explanation,
    })

    # ---- printing: this is the part that changed for R3 ----
    print(f"--- {row['user_id']} ---")
    print(f"risk:         {decision['risk']:.2f}")
    print(f"action:       {decision['action']}")
    print(f"explanation:  {explanation}")
    if decision["action"] == "hold_and_alert":
        print(f"family alert: {family_alert(row)}")
    print()

results = pd.DataFrame(results)

--- rosa ---
risk:         0.95
action:       hold_and_alert
explanation:  stub explanation - real version coming
family alert: SafeCircle paused a $1,500 gift-card purchase on Rosa's card after a likely scam text. Please call Rosa.

--- walter ---
risk:         0.45
action:       allow
explanation:  stub explanation - real version coming

--- linda ---
risk:         0.91
action:       hold_and_alert
explanation:  stub explanation - real version coming
family alert: SafeCircle paused a $300 wire transfer on Linda's card after a likely scam text. Please call Linda.



In [10]:
results["result"] = results.apply(
    lambda r: "PASS" if r["action"] == r["expected_action"] else "FAIL", axis=1)

for _, r in results.iterrows():
    print(f"{r['result']}  {r['user_id']:8} expected={r['expected_action']:15} got={r['action']}")

print(f"\n{(results['result'] == 'PASS').sum()} / {len(results)} scenarios pass")

PASS  rosa     expected=hold_and_alert  got=hold_and_alert
PASS  walter   expected=allow           got=allow
PASS  linda    expected=hold_and_alert  got=hold_and_alert

3 / 3 scenarios pass
